# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [ ]:
# try:
#     !pip install gurobipy
# except:
#     %pip install gurobipy
# test von Ben

In [28]:
!git clone -b feature/shift-objects https://github.com/poprjaduhhaa/Modellierungsseminar-Firestation.git
%cd /content/Modellierungsseminar-Firestation/coding

[WinError 3] Das System kann den angegebenen Pfad nicht finden: '/content/Modellierungsseminar-Firestation/coding'
c:\Users\dirkb\Documents\GitHub\Modellierungsseminar-Firestation\coding


fatal: destination path 'Modellierungsseminar-Firestation' already exists and is not an empty directory.


In [29]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions

Note: you may need to restart the kernel to use updated packages.


### inputs and parameters

In [30]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(52/4)  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }

DICT_WEEKDAYS_RETURN = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}

# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs"

In [ ]:
# LOGS / for process transparency 
def writeToLogs(yourStatusMessage:str):
    with FOLDER_AND_FILE_LOG.open("a") as log:
        log.write(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        log.write(" // ")
        log.write(yourStatusMessage+"\n")

writeToLogs("STARTED the cycle planning process")

In [32]:

# basic inputs and parameters

Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday
#Shifts = ["frueh", "spaet", "nacht", "frei"] # including free shifts
#WorkShifts = ["frueh", "spaet", "nacht"] # excluding free shifts

# improvements outstanding:
    # use files for parameter input
        # shift definitions
        # available staff
        # user objectives: weighted priorities

In [33]:
# read input data
# shift set

def readShiftSet(filename, mySep: str=";") -> pd.DataFrame:
    writeToLogs(f"reading shift data set from file {filename}")
    input_data = pd.read_csv(filename, sep=mySep, dtype=str) # import all values as string as first step
    # adjust data types for columns not (supposed to be) reflecting strings
    input_data["shift_required_staff"].astype(int)
    input_data["shift_class"].astype(int)
    input_data["shift_work_time_assignment"].astype(float)
    input_data["isWorkShift"] = input_data["isWorkShift"].astype(int).astype(bool)
    # multiply rows by the number of required workers (.explode())
    #input_data = input_data.assign(shift_required_staff=input_data["shift_required_staff"].apply(lambda n: list(range(1, n+1)))).explode("shift_required_staff")

    writeToLogs("import successfull")
    return input_data
    # potentially add further data cleaning steps


In [34]:

# added objects to read whole csv file
def build_shift_objects(df: pd.DataFrame) -> list:
    shift_objects = []
    for _, row in df.iterrows():
        s = Shift.Shift(
            shift_id=row['shift_ID'],
            description=row['shift_details'],
            weekdays=[d.strip() for d in row['shift_weekdays'].split(',')],
            start=dt.time(*map(int, row['shift_start_time'].split(':'))),
            end=dt.time(*map(int, row['shift_end_time'].replace('24','0').split(':'))),
            required_staff=0 if row['shift_required_staff'] == 'none' else int(row['shift_required_staff']),
            shift_class=int(row['shift_class']),
            shift_work_time_assignment=str(row['shift_work_time_assignment']),
            is_work_shift=bool(row['isWorkShift']),
            required_qualification=row['[shift_required_qualification]']
        )
        shift_objects.append(s)
    writeToLogs("list of shift_objects built")
    return shift_objects


creating shift objects:

In [ ]:

# folderpath = FOLDER_INPUT
# filename = "input_ShiftDataSet_Pesch.csv"

data_shiftSet = readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")
#data_shiftSet["isWorkShift"] = data_shiftSet["isWorkShift"].astype(int).astype(bool) # this step was moved to import function
#print(data_shiftSet)

shift_objects = build_shift_objects(data_shiftSet)
freeDayShift = Shift.Shift("[freeDay]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           None, 
                           1, 
                           None, 
                           False, 
                           None 
                           )
shift_objects.append(freeDayShift)
# LOG / for transparency write shift objects into a file:
try:
    pd.DataFrame(shift_objects).to_csv(FOLDER_LOGS / "log_shift_object.csv", sep=";", index=True, encoding="utf-8", decimal=".")
except PermissionError:
    print("log file for shift_object is open - I skipped saving and executed succeeding code")
# for s in shift_objects:
#     print(s)

Shifts = [s.shift_id for s in shift_objects]
WorkShifts = [s.shift_id for s in shift_objects if s.is_work_shift] #object oriented solution

#Shifts = list(data_shiftSet["shift_ID"]) # including free shifts
#WorkShifts = list(data_shiftSet[data_shiftSet["isWorkShift"]]["shift_ID"]) # excluding free shifts


#print(Shifts)
#print(data_shiftSet["shift_weekdays"])



### modelling

In [36]:
# modelling

m = gp.Model("SnakeBuilding_simple")

# variables:
# x[s, d, sh] = 1, when snake s is working in shift sh on day d
x = m.addVars(MAX_CYCLE_WEEKS, Weekdays, Shifts, vtype=GRB.BINARY, name="x")

# active[s] = 1, when snake s is used
active = m.addVars(MAX_CYCLE_WEEKS, vtype=GRB.BINARY, name="active") # all other snakes are used as placeholders but not necessarily get activated


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806446
Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [37]:
# SUBJECT TO:

# 1. each shift has to be covered on each day
for d in Weekdays:
    for ws in WorkShifts:
        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
                    name=f"Cover_day{d}_{ws}")


####

#### condition c02

In [38]:
# 2. each snake can have at most one shift per day
for s in range(MAX_CYCLE_WEEKS):
    for d in Weekdays:
        m.addConstr(gp.quicksum(x[s, d, sh] for sh in Shifts) == active[s],
                    name=f"OneShiftPerDay_s{s}_d{d}")


#### condition c03

In [39]:

# 3) at max 5 consecutive working days (ensure time for resting)
for s in range(MAX_CYCLE_WEEKS):
    for start in range(1, 7-5+1):
        m.addConstr(
            gp.quicksum(x[s, d, sh] for d in range(start, start + 6)
                        for sh in WorkShifts) <= 5,
            name=f"Max5Work_s{s}_start{start}"
        )



#### condition c04  

ensure that cycle weeks are activated in ascending order  

$$y_s - y_{s+1} >= 0$$

In [40]:
# 4) ensure that cycle weeks are activated in ascending order (not like 2-5-9-17-29-.... but 1-2-3-4-....)

for s in range(MAX_CYCLE_WEEKS-1):
    m.addConstr(active[s]>=active[s+1])

### objective

In [41]:
# set objective function: minimize number of active snakes
m.setObjective(gp.quicksum(active[s] for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives => based on user input

#run optimizer
m.optimize()



Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 PRO 250 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 157 rows, 468 columns and 1558 nonzeros (Min)
Model fingerprint: 0x20728f3c
Model has 13 linear objective coefficients
Variable types: 0 continuous, 468 integer (468 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Found heuristic solution: objective 13.0000000
Presolve removed 32 rows and 145 columns
Presolve time: 0.00s
Presolved: 125 rows, 323 columns, 1022 nonzeros
Variable types: 0 continuous, 323 integer (323 binary)

Root relaxation: objective 4.333333e+00, 192 iterations, 0.00 seconds (0.

### results

In [42]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file
        # create a shift overview per staff member

if m.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(m.objVal))
    writeToLogs(f"successfully finished cycle plan: found an optimal solution using {int(m.objVal)} cycle weeks")
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X == 1:
            print(f"\ncycle week {s+1}:")
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X == 1:
                        print(f"  day {d}: {sh}")



minimum number of cycle weeks: 5

cycle week 1:
  day 1: [night000week]
  day 2: [night000week]
  day 3: [00day000week]
  day 4: [00day000week]
  day 5: [nightweekend]
  day 6: [00000freeday]
  day 7: [00day000week]

cycle week 2:
  day 1: [00dayweekend]
  day 2: [00dayweekend]
  day 3: [night000week]
  day 4: [00000freeday]
  day 5: [night000week]
  day 6: [night000week]
  day 7: [night000week]

cycle week 3:
  day 1: [00day000week]
  day 2: [00000freeday]
  day 3: [00dayweekend]
  day 4: [nightweekend]
  day 5: [00day000week]
  day 6: [00dayweekend]
  day 7: [nightweekend]

cycle week 4:
  day 1: [00000freeday]
  day 2: [nightweekend]
  day 3: [nightweekend]
  day 4: [00dayweekend]
  day 5: [00dayweekend]
  day 6: [00day000week]
  day 7: [00000freeday]

cycle week 5:
  day 1: [nightweekend]
  day 2: [00day000week]
  day 3: [00000freeday]
  day 4: [night000week]
  day 5: [00000freeday]
  day 6: [nightweekend]
  day 7: [00dayweekend]


In [43]:
print(shift_objects[1])

Shift(shift_id='[night000week]', description='night shift week', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'], start=datetime.time(16, 15), end=datetime.time(7, 0), required_staff=5, shift_class=7, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
